# ⚡ Inference & Model Optimization — NVIDIA Senior DL Software Engineer Role

**Focus:** LLM / diffusion model deployment, quantization, pruning, GPU kernel profiling  
**Stack:** PyTorch 2.0, torch.compile, CUDA events, TensorRT concepts, Triton  
**Role:** Senior Deep Learning Software Engineer — Inference and Model Optimization

## What this role does
NVIDIA's inference engineering team takes research models (LLMs, diffusion models, vision transformers)
and makes them fast enough to deploy at scale — on data-center GPUs and edge devices alike.
This means:

1. **Model compression** — quantization, pruning, sparsity so models fit in memory and run faster
2. **Compiler optimization** — torch.compile, TensorRT, kernel fusion to maximize GPU utilization
3. **Profiling** — find the bottleneck ops, analyze roofline, optimize memory bandwidth
4. **Scalable deployment** — tensor/pipeline parallelism across multiple GPUs

| What you optimize | Why it's hard |
|---|---|
| Quantization to INT8/FP16 | Accuracy degrades if done naively |
| Pruning weights | Sparsity patterns must match hardware for speedup |
| Kernel fusion | Graph compilers must handle control flow and dynamic shapes |
| Multi-GPU sharding | Communication overhead can exceed compute savings |
| Latency vs throughput | Batching improves throughput but hurts latency |

---

In this notebook you will:
1. Understand the **inference optimization pipeline** from model to deployment
2. Capture and inspect **PyTorch 2.0 computation graphs** (torch.fx, torch.export)
3. Implement **quantization** (INT8/FP16) from scratch and measure accuracy vs memory
4. Apply **magnitude pruning** and measure the sparsity/accuracy tradeoff
5. **Profile GPU kernels** with CUDA events and analyze the roofline model
6. **Benchmark inference** — latency vs throughput, batch size sweeps, eager vs compiled
7. Understand **model sharding** for multi-GPU deployment
8. Complete **3 challenges** that reflect real senior DL engineer work at NVIDIA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
import numpy as np
import matplotlib.pyplot as plt
import cupy as cp
import time
import math
import copy
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

print("✅ Imports OK")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   GPU : {gpu_name}")
    print(f"   VRAM: {total_mem:.1f} GB")
    print(f"   CUDA: {torch.version.cuda}")
    print(f"   PyTorch: {torch.__version__}")
else:
    print("   ⚠️  No CUDA GPU found — some cells will run on CPU")

## 🗺️ Intro: The Inference Optimization Landscape

Before a model reaches users, it goes through a multi-stage optimization pipeline:

```
┌─────────────────────────────────────────────────────────────────────┐
│                  INFERENCE OPTIMIZATION PIPELINE                    │
└─────────────────────────────────────────────────────────────────────┘

  ┌──────────┐     ┌──────────┐     ┌─────────────┐     ┌──────────────┐
  │          │     │          │     │             │     │              │
  │  Trained │────▶│ Pruning  │────▶│Quantization │────▶│ Compilation  │
  │  Model   │     │ Sparsity │     │ INT8/FP16   │     │torch.compile │
  │  (FP32)  │     │          │     │             │     │  TensorRT    │
  └──────────┘     └──────────┘     └─────────────┘     └──────┬───────┘
                                                                │
                                                                ▼
                                                       ┌──────────────┐
                                                       │  Deployment  │
                                                       │              │
                                                       │ Single GPU   │
                                                       │ Tensor Para. │
                                                       │ Pipeline Par.│
                                                       └──────────────┘

  Each stage trades a small accuracy loss for large gains in speed or memory.
```

### NVIDIA Tools × Optimization Techniques

| Technique | NVIDIA Tool | Typical Speedup | Accuracy Impact |
|---|---|---|---|
| FP16 mixed precision | `torch.cuda.amp` | 1.5–2×  | Negligible |
| INT8 quantization | TensorRT, `torch.ao.quantization` | 2–4× | Small (0.1–1%) |
| INT4 quantization | TensorRT-LLM, AWQ, GPTQ | 3–6× | Moderate |
| Magnitude pruning | `torch.nn.utils.prune` | 1–2× (unstructured) | Small–moderate |
| Structured pruning | Custom / APEX | 1.5–3× | Moderate |
| Kernel fusion | `torch.compile`, TensorRT | 1.2–2× | None |
| Tensor parallelism | Megatron-LM, NeMo | Scales with GPUs | None |
| Speculative decoding | TensorRT-LLM | 2–3× (LLMs) | None |
| Flash Attention | `F.scaled_dot_product_attention` | 2–4× (attention) | None |

### Why this role is technically demanding
Each optimization has a different failure mode. Quantization can cause catastrophic
accuracy collapse on certain layers (embedding tables, layer norms). Pruning can
shatter learned representations if applied too aggressively. Kernel fusion can
introduce numerical differences. This engineer must navigate all of these tradeoffs
simultaneously, at the level of individual GPU kernels.

## 1. 🔍 PyTorch 2.0 Model Representations

Before optimizing a model, you need a programmatic representation of its computation
graph. PyTorch 2.0 introduced a standardized IR (intermediate representation) stack:

```
  Python Model (nn.Module)
          │
          ▼  torch.fx.symbolic_trace  (eager-mode graph capture)
      fx.Graph  ←──── operations as nodes: call_function, call_module, placeholder
          │
          ▼  torch.export  (strict, serializable, no Python side-effects)
   ExportedProgram  ←── ExportGraph + state_dict + constraints
          │
          ▼  torch.compile (TorchDynamo + TorchInductor backend)
   Compiled artifact  ←── Triton kernels / CUDA C++ / cuBLAS calls
```

### torch.fx — symbolic tracing
`torch.fx.symbolic_trace` runs the model with symbolic inputs ("proxy" tensors)
and records every operation into a `Graph` object. You can then inspect, transform,
and re-compile the graph. This is the foundation of quantization, pruning tools,
and operator fusion.

### torch.export — strict export
`torch.export.export` goes further: it enforces that the model has no data-dependent
control flow that would prevent AOT (ahead-of-time) compilation. The result is a
fully serializable `ExportedProgram` that can be fed to TensorRT or ExecuTorch.

In [ ]:
# ── Define a small toy model ───────────────────────────────────────────────────
class TinyMLP(nn.Module):
    """A small MLP — representative of a feed-forward block inside a transformer."""
    def __init__(self, d_in=64, d_hidden=256, d_out=64):
        super().__init__()
        self.fc1  = nn.Linear(d_in, d_hidden)
        self.act  = nn.GELU()
        self.fc2  = nn.Linear(d_hidden, d_out)
        self.norm = nn.LayerNorm(d_out)

    def forward(self, x):
        return self.norm(self.fc2(self.act(self.fc1(x))))


model = TinyMLP().eval()

# ── Capture the computation graph with torch.fx ────────────────────────────────
import torch.fx as fx

traced: fx.GraphModule = fx.symbolic_trace(model)

print("=" * 60)
print("torch.fx computation graph — TinyMLP")
print("=" * 60)
print(f"{'Node name':<28} {'Op':<18} {'Target':<25}")
print("-" * 75)
for node in traced.graph.nodes:
    target_str = str(node.target)[:24] if node.target else ""
    print(f"{node.name:<28} {node.op:<18} {target_str:<25}")

print()
n_nodes = len(list(traced.graph.nodes))
n_params = sum(p.numel() for p in model.parameters())
print(f"Graph nodes  : {n_nodes}")
print(f"Model params : {n_params:,}")
print(f"Param bytes  : {n_params * 4 / 1024:.1f} KB  (FP32)")

In [ ]:
# ── torch.export — strict exportable program ───────────────────────────────────
# torch.export requires concrete example inputs and produces a fully
# serialisable ExportedProgram with no Python side-effects.

example_input = (torch.randn(4, 64),)   # batch=4, d_in=64

with torch.no_grad():
    exported = torch.export.export(model, example_input)

print("=" * 60)
print("torch.export.ExportedProgram")
print("=" * 60)

# Show the exported graph nodes
print(f"\nGraph signature inputs  : {[str(s) for s in exported.graph_signature.input_specs[:3]]} ...")
print(f"Graph signature outputs : {[str(s) for s in exported.graph_signature.output_specs]}")

print("\nExported graph nodes:")
print(f"{'Name':<30} {'Op':<20} {'Target'}")
print("-" * 70)
for node in exported.graph.nodes:
    tgt = str(node.target)[:35] if node.target else ""
    print(f"{node.name:<30} {node.op:<20} {tgt}")

# Verify correctness — exported program must produce same output as original
x_test = torch.randn(4, 64)
with torch.no_grad():
    out_original = model(x_test)
    out_exported = exported.module()(x_test)

max_diff = (out_original - out_exported).abs().max().item()
print(f"\nMax output diff (original vs exported): {max_diff:.2e}  ", end="")
print("✅ OK" if max_diff < 1e-5 else "❌ MISMATCH")

## 2. 🔢 Quantization — INT8 and FP16

**Quantization** maps floating-point weights/activations to a lower-precision format
(INT8, INT4, FP16) to save memory and accelerate matrix multiplications on GPU
hardware that has dedicated low-precision compute units (Tensor Cores).

### Quantization math

For **uniform symmetric quantization** to INT8 (range −128 to 127):

```
  scale     = max(|x|) / 127
  x_quant   = clamp( round(x / scale), -128, 127 )    ← integer
  x_dequant = x_quant * scale                          ← back to float

  quantization error = x_dequant - x
```

For **affine (asymmetric) quantization** (used for activations, which are often non-negative):
```
  scale      = (max(x) - min(x)) / 255
  zero_point = round(-min(x) / scale)          ← integer offset
  x_quant    = clamp( round(x / scale) + zero_point, 0, 255 )   ← UINT8
  x_dequant  = (x_quant - zero_point) * scale
```

### Post-Training Quantization (PTQ) vs Quantization-Aware Training (QAT)

| Method | When | Accuracy | Speed |
|---|---|---|---|
| **PTQ symmetric** | After training | Good for weights | Easiest |
| **PTQ affine** | After training | Better for activations | Easy |
| **QAT** | During training | Best | Requires retraining |
| **GPTQ / AWQ** | After training (LLM) | Good for INT4 | Medium |

### Memory savings
- FP32 → FP16: **2×** smaller
- FP32 → INT8: **4×** smaller
- FP32 → INT4: **8×** smaller

In [ ]:
# ── Manual quantization implementation ────────────────────────────────────────
# Implementing this from scratch is a common interview topic and helps you
# understand what libraries like TensorRT are doing internally.

def quantize_symmetric_int8(tensor: torch.Tensor) -> Tuple[torch.Tensor, float]:
    """
    Symmetric INT8 quantization.
    Returns (quantized_int8_tensor, scale).
    """
    max_val = tensor.abs().max().item()
    scale   = max_val / 127.0
    if scale == 0:
        scale = 1e-8
    q = torch.clamp(torch.round(tensor / scale), -128, 127).to(torch.int8)
    return q, scale


def dequantize_symmetric_int8(q: torch.Tensor, scale: float) -> torch.Tensor:
    """Restore FP32 from INT8 + scale."""
    return q.to(torch.float32) * scale


def quantize_affine_uint8(tensor: torch.Tensor) -> Tuple[torch.Tensor, float, int]:
    """
    Affine (asymmetric) UINT8 quantization.
    Returns (quantized_uint8_tensor, scale, zero_point).
    """
    x_min = tensor.min().item()
    x_max = tensor.max().item()
    scale  = (x_max - x_min) / 255.0
    if scale == 0:
        scale = 1e-8
    zero_point = int(round(-x_min / scale))
    zero_point = max(0, min(255, zero_point))
    q = torch.clamp(torch.round(tensor / scale) + zero_point, 0, 255).to(torch.uint8)
    return q, scale, zero_point


def dequantize_affine_uint8(q: torch.Tensor, scale: float, zero_point: int) -> torch.Tensor:
    return (q.to(torch.float32) - zero_point) * scale


# ── Test on a realistic weight matrix ─────────────────────────────────────────
torch.manual_seed(42)
# Simulate the weight matrix of a transformer's attention projection
W_fp32 = torch.randn(256, 256) * 0.02   # typical Kaiming init scale

# --- INT8 symmetric (good for weights, which are usually symmetric around 0) ---
W_q8,  scale8              = quantize_symmetric_int8(W_fp32)
W_deq8                     = dequantize_symmetric_int8(W_q8, scale8)

# --- UINT8 affine (better for activations like ReLU output, which are ≥ 0) ---
A_fp32 = torch.relu(torch.randn(256, 256) * 0.5)  # relu activations
A_qu8, scale_a, zp_a       = quantize_affine_uint8(A_fp32)
A_deq8                     = dequantize_affine_uint8(A_qu8, scale_a, zp_a)

# --- FP16 (half precision — no explicit quantization, just dtype cast) ---
W_fp16 = W_fp32.half()
W_defp16 = W_fp16.float()

# ── Accuracy metrics ──────────────────────────────────────────────────────────
def quant_error_stats(original, dequantized, name):
    err = (original - dequantized).abs()
    snr = 10 * math.log10(original.pow(2).mean().item() / (err.pow(2).mean().item() + 1e-12))
    print(f"  {name:30s}  MAE={err.mean():.6f}  MaxErr={err.max():.6f}  SNR={snr:.1f} dB")

print("Quantization error analysis:")
print("-" * 75)
quant_error_stats(W_fp32,  W_deq8,   "Weights  INT8 symmetric")
quant_error_stats(W_fp32,  W_defp16, "Weights  FP16")
quant_error_stats(A_fp32,  A_deq8,   "Activations UINT8 affine")

# ── Memory comparison ─────────────────────────────────────────────────────────
print()
print("Memory footprint (256×256 matrix):")
print(f"  FP32   : {W_fp32.numel() * 4 / 1024:.1f} KB")
print(f"  FP16   : {W_fp32.numel() * 2 / 1024:.1f} KB  ({4/2:.0f}× smaller)")
print(f"  INT8   : {W_fp32.numel() * 1 / 1024:.1f} KB  ({4/1:.0f}× smaller)")
print(f"  INT4   : {W_fp32.numel() * 0.5 / 1024:.1f} KB  ({4/0.5:.0f}× smaller) [conceptual]")

In [ ]:
# ── End-to-end: quantize a full TinyMLP and compare inference accuracy ─────────

class QuantizedLinear(nn.Module):
    """
    A drop-in replacement for nn.Linear that quantizes weights to INT8
    at module creation time (post-training quantization).
    Forward pass dequantizes before the matmul ("fake quantization" — simulates
    INT8 error without requiring actual INT8 hardware support).
    """
    def __init__(self, linear: nn.Linear):
        super().__init__()
        W = linear.weight.data
        self.bias = linear.bias

        # Quantize weights at construction (offline, zero runtime cost)
        q, s = quantize_symmetric_int8(W)
        self.register_buffer("weight_q8", q)
        self.scale = s
        self.out_features, self.in_features = W.shape

    def forward(self, x):
        # Dequantize on the fly (simulates INT8 weight-only quantization)
        W_deq = dequantize_symmetric_int8(self.weight_q8, self.scale)
        return F.linear(x, W_deq, self.bias)


def quantize_model(model: nn.Module) -> nn.Module:
    """Replace all nn.Linear layers with QuantizedLinear."""
    model_q = copy.deepcopy(model)
    for name, module in list(model_q.named_children()):
        if isinstance(module, nn.Linear):
            setattr(model_q, name, QuantizedLinear(module))
    return model_q


torch.manual_seed(0)
model_fp32 = TinyMLP(d_in=64, d_hidden=256, d_out=64).eval()
model_int8 = quantize_model(model_fp32).eval()

# ── Compare outputs on 1000 random inputs ─────────────────────────────────────
errors = []
torch.manual_seed(7)
for _ in range(200):
    x = torch.randn(8, 64)
    with torch.no_grad():
        out_fp32 = model_fp32(x)
        out_int8 = model_int8(x)
    errors.append((out_fp32 - out_int8).abs().mean().item())

errors = np.array(errors)
print("INT8 quantized model vs FP32 — output error statistics:")
print(f"  Mean absolute error : {errors.mean():.6f}")
print(f"  Max  absolute error : {errors.max():.6f}")
print(f"  Std                 : {errors.std():.6f}")

# ── Memory footprint ──────────────────────────────────────────────────────────
def param_bytes(m):
    total = 0
    for name, buf in m.named_buffers():
        total += buf.numel() * buf.element_size()
    for name, p in m.named_parameters():
        total += p.numel() * p.element_size()
    return total

fp32_bytes = param_bytes(model_fp32)
int8_bytes = param_bytes(model_int8)
print()
print(f"FP32 model size : {fp32_bytes / 1024:.1f} KB")
print(f"INT8 model size : {int8_bytes / 1024:.1f} KB  ({fp32_bytes/int8_bytes:.2f}× smaller)")

In [ ]:
# ── Visualise quantization error distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Quantization Analysis — Symmetric INT8", fontsize=13, fontweight="bold")

# Left: weight value histogram — FP32 vs dequantized INT8
ax = axes[0]
ax.set_title("Weight Distribution (fc1)")
W = model_fp32.fc1.weight.data.flatten().numpy()
W_q, sc = quantize_symmetric_int8(torch.tensor(W))
W_dq = dequantize_symmetric_int8(W_q, sc).numpy()
ax.hist(W,   bins=60, alpha=0.6, color="steelblue", label="FP32")
ax.hist(W_dq, bins=60, alpha=0.6, color="tomato",    label="INT8 dequant")
ax.set_xlabel("Weight value")
ax.set_ylabel("Count")
ax.legend()

# Middle: quantization error histogram
ax2 = axes[1]
ax2.set_title("Quantization Error (fc1 weights)")
err_w = W - W_dq
ax2.hist(err_w, bins=60, color="#e67e22", alpha=0.8)
ax2.axvline(err_w.mean(), color="red", linestyle="--", label=f"Mean={err_w.mean():.5f}")
ax2.set_xlabel("FP32 - INT8_dequant")
ax2.set_ylabel("Count")
ax2.legend()

# Right: output error distribution across 200 batches
ax3 = axes[2]
ax3.set_title("Output MAE over 200 Batches")
ax3.plot(errors, color="#8e44ad", alpha=0.7, lw=0.9)
ax3.axhline(errors.mean(), color="red", linestyle="--", label=f"Mean MAE={errors.mean():.5f}")
ax3.set_xlabel("Batch #")
ax3.set_ylabel("Mean absolute error")
ax3.legend()

plt.tight_layout()
plt.show()

## 3. ✂️ Pruning and Sparsity

**Pruning** removes weights from a neural network that contribute little to the output.
The goal is to create a **sparse** model that is smaller and (with the right hardware support)
faster at inference.

### Structured vs Unstructured Pruning

```
  UNSTRUCTURED PRUNING              STRUCTURED PRUNING
  ─────────────────────             ──────────────────────
  Remove individual weights         Remove entire rows / columns

  ┌───────────────────┐             ┌───────────────────┐
  │ W W 0 W 0 0 W W  │             │ W W W W W W W W  │  ← keep row
  │ 0 W W 0 W 0 W 0  │             │ 0 0 0 0 0 0 0 0  │  ← pruned row
  │ W 0 0 W W 0 0 W  │             │ W W W W W W W W  │
  │ 0 W 0 0 0 W W 0  │             │ 0 0 0 0 0 0 0 0  │  ← pruned row
  └───────────────────┘             └───────────────────┘
    Sparse — needs special           Dense — works with
    hardware (NVIDIA A100            standard cuBLAS.
    2:4 sparsity) for speedup.       Real speedup on any GPU.
```

### Magnitude-based pruning
The simplest strategy: remove weights with smallest absolute value.
Rationale: small weights contribute least to layer outputs (approximately).

### NVIDIA 2:4 structured sparsity
The A100/H100 Tensor Core has a special mode where exactly 2 of every 4 consecutive
weights are non-zero. This gives exactly **2× speedup** on GEMM with no accuracy loss
compared to unstructured 50% sparsity. `torch.ao.pruning` supports this pattern.

In [ ]:
# ── Train a small MLP on random classification data ───────────────────────────
# We need a trained model so pruning actually has an accuracy impact to measure.

torch.manual_seed(42)
np.random.seed(42)

N_TRAIN, N_TEST = 2000, 500
D_IN, D_HIDDEN, N_CLASSES = 32, 128, 4

# Random linearly-separable classification task
X_train = torch.randn(N_TRAIN, D_IN)
W_true  = torch.randn(D_IN, N_CLASSES) * 0.5
y_train = (X_train @ W_true).argmax(dim=1)
X_test  = torch.randn(N_TEST, D_IN)
y_test  = (X_test @ W_true).argmax(dim=1)


class ClassifierMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D_IN, D_HIDDEN),
            nn.ReLU(),
            nn.Linear(D_HIDDEN, D_HIDDEN),
            nn.ReLU(),
            nn.Linear(D_HIDDEN, N_CLASSES),
        )

    def forward(self, x):
        return self.net(x)


def train_model(model, epochs=60):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        logits = model(X_train)
        loss = F.cross_entropy(logits, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()


def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()


def sparsity(model):
    zeros = total = 0
    for p in model.parameters():
        zeros += (p.data == 0).sum().item()
        total += p.numel()
    return zeros / total


baseline_model = ClassifierMLP()
train_model(baseline_model)
baseline_acc = accuracy(baseline_model, X_test, y_test)
print(f"Baseline accuracy (dense, 0% pruned): {baseline_acc*100:.1f}%")

# ── Sparsity vs accuracy sweep ─────────────────────────────────────────────────
prune_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
accs, sparsities = [], []

for amount in prune_levels:
    m = copy.deepcopy(baseline_model)

    if amount > 0:
        # Apply global unstructured magnitude pruning across all Linear layers
        params_to_prune = [
            (layer, "weight")
            for layer in m.modules()
            if isinstance(layer, nn.Linear)
        ]
        prune.global_unstructured(
            params_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=amount,
        )
        # Make pruning permanent (remove mask buffers)
        for layer, _ in params_to_prune:
            prune.remove(layer, "weight")

    acc = accuracy(m, X_test, y_test)
    sp  = sparsity(m)
    accs.append(acc)
    sparsities.append(sp)
    print(f"  Pruning amount={amount:.0%}  actual sparsity={sp:.1%}  accuracy={acc*100:.1f}%")

print()
print("Key insight: accuracy remains high until ~50–60% pruning, then degrades rapidly.")

In [ ]:
# ── Visualise sparsity / accuracy tradeoff ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Pruning Analysis — Magnitude-Based Unstructured Pruning", fontsize=13, fontweight="bold")

# Left: accuracy vs sparsity curve
ax = axes[0]
ax.set_title("Accuracy vs Sparsity")
ax.plot([s * 100 for s in sparsities], [a * 100 for a in accs],
        marker="o", color="#2980b9", lw=2)
ax.axhline(baseline_acc * 100, color="green", linestyle="--", label=f"Baseline {baseline_acc*100:.1f}%")
ax.axhline(baseline_acc * 100 - 2.0, color="orange", linestyle=":", label="−2% tolerance")
ax.set_xlabel("Actual Sparsity (%)")
ax.set_ylabel("Test Accuracy (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: weight magnitude histogram of dense vs pruned model
ax2 = axes[1]
ax2.set_title("Weight Magnitudes — Dense vs 50% Pruned")
dense_weights = torch.cat([p.data.flatten() for p in baseline_model.parameters()]).numpy()

m50 = copy.deepcopy(baseline_model)
params_to_prune = [(l, "weight") for l in m50.modules() if isinstance(l, nn.Linear)]
prune.global_unstructured(params_to_prune, pruning_method=prune.L1Unstructured, amount=0.5)
for l, _ in params_to_prune:
    prune.remove(l, "weight")
pruned_weights = torch.cat([p.data.flatten() for p in m50.parameters()]).numpy()

ax2.hist(dense_weights,  bins=80, alpha=0.6, color="steelblue", label="Dense")
ax2.hist(pruned_weights, bins=80, alpha=0.6, color="tomato",    label="50% pruned")
ax2.set_xlabel("Weight value")
ax2.set_ylabel("Count")
ax2.legend()
ax2.set_yscale("log")

plt.tight_layout()
plt.show()

## 4. 🔬 GPU Kernel Performance Analysis

After compression, you need to know *where* the remaining compute time goes.
NVIDIA engineers use the **roofline model** to understand whether an operation is
compute-bound or memory-bandwidth-bound.

### The Roofline Model

```
  Attainable FLOPS/s
  │                                          ┌──────────────────────
  │                                         /  Peak compute (FLOPS)
  │                                        /   RTX 4070 SUPER: ~48 TFLOPS FP32
  │                                       /    RTX 4070 SUPER: ~386 TFLOPS FP16
  │        memory-bandwidth bound         /
  │       (operation hits mem wall)      / ← "ridge point"
  │                                     /
  │         slope = mem bandwidth       /  compute bound
  │         RTX 4070 SUPER: ~432 GB/s  /   (sits on flat line)
  │                                   /
  └──────────────────────────────────────────────────────────────▶
                                      Arithmetic Intensity
                                      (FLOPS / byte accessed)

  Low AI (few FLOPS per byte) → memory bound → improve cache reuse / bandwidth
  High AI (many FLOPS per byte) → compute bound → reduce FLOP count / fuse kernels

  Examples:
    Elementwise ReLU : AI ≈ 0.25 FLOP/byte  → strongly memory bound
    GEMM (large N)   : AI ≈ N/2  FLOP/byte  → compute bound for large N
    Attention (QKV)  : AI ≈ seq_len/8       → memory bound for short sequences
```

### CUDA events for accurate timing
CPU timers (`time.time()`) measure wall-clock time including Python overhead and
GPU launch latency. `torch.cuda.Event` records timestamps on the GPU stream itself,
giving accurate kernel execution time.

In [ ]:
# ── GPU kernel profiler using CUDA events ─────────────────────────────────────

@dataclass
class KernelProfile:
    name: str
    times_ms: List[float]
    flops: int           # theoretical FLOPs for this operation
    bytes_accessed: int  # theoretical bytes read+written

    @property
    def median_ms(self):  return float(np.median(self.times_ms))
    @property
    def mean_ms(self):    return float(np.mean(self.times_ms))
    @property
    def std_ms(self):     return float(np.std(self.times_ms))
    @property
    def tflops(self):     return self.flops / (self.median_ms * 1e-3) / 1e12
    @property
    def bandwidth_gbs(self): return self.bytes_accessed / (self.median_ms * 1e-3) / 1e9
    @property
    def arithmetic_intensity(self): return self.flops / self.bytes_accessed

    def report(self):
        print(f"  {self.name}")
        print(f"    Median latency      : {self.median_ms:.4f} ms")
        print(f"    Std dev             : {self.std_ms:.4f} ms")
        print(f"    Throughput          : {self.tflops:.2f} TFLOPS")
        print(f"    Memory bandwidth    : {self.bandwidth_gbs:.1f} GB/s")
        print(f"    Arithmetic intensity: {self.arithmetic_intensity:.2f} FLOP/byte")


def profile_op(name, fn, flops, bytes_accessed, n_warmup=20, n_runs=100):
    """Profile a GPU op with CUDA events."""
    # Warmup
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))

    return KernelProfile(name=name, times_ms=times, flops=flops, bytes_accessed=bytes_accessed)


# ── Profile three representative ops ──────────────────────────────────────────
M, N, K = 2048, 2048, 2048
A = torch.randn(M, K, device="cuda", dtype=torch.float32)
B = torch.randn(K, N, device="cuda", dtype=torch.float32)

# GEMM: 2*M*N*K FLOPs, reads A(M*K) + B(K*N), writes C(M*N)
prof_gemm = profile_op(
    "GEMM (2048×2048×2048, FP32)",
    fn=lambda: torch.mm(A, B),
    flops=2 * M * N * K,
    bytes_accessed=(M*K + K*N + M*N) * 4,
)

# Conv2d: ~2 * Cout * Cin * kH * kW * H * W FLOPs
x_img = torch.randn(16, 64, 56, 56, device="cuda")
conv  = nn.Conv2d(64, 128, 3, padding=1).cuda()
Cout, Cin, kH, kW, H, W, B_sz = 128, 64, 3, 3, 56, 56, 16
conv_flops = 2 * B_sz * Cout * Cin * kH * kW * H * W
conv_bytes = (B_sz * Cin * H * W + Cout * Cin * kH * kW + B_sz * Cout * H * W) * 4
prof_conv = profile_op(
    "Conv2d (16×64×56×56 → 128 ch, 3×3)",
    fn=lambda: conv(x_img),
    flops=conv_flops,
    bytes_accessed=conv_bytes,
)

# Scaled dot-product attention (PyTorch's FlashAttention-backed implementation)
S, L, H_heads, D_head = 8, 512, 8, 64   # batch, seq_len, heads, head_dim
q = torch.randn(S, H_heads, L, D_head, device="cuda")
k = torch.randn(S, H_heads, L, D_head, device="cuda")
v = torch.randn(S, H_heads, L, D_head, device="cuda")
# FLOPs: 4 * B * H * L^2 * D (QK^T matmul + softmax + AV matmul, approx)
attn_flops = 4 * S * H_heads * L * L * D_head
attn_bytes = (3 * S * H_heads * L * D_head + S * H_heads * L * D_head) * 4
prof_attn = profile_op(
    "Flash Attention (B=8, H=8, L=512, D=64)",
    fn=lambda: F.scaled_dot_product_attention(q, k, v),
    flops=attn_flops,
    bytes_accessed=attn_bytes,
)

print("Kernel profiling results:")
print("=" * 60)
for prof in [prof_gemm, prof_conv, prof_attn]:
    prof.report()
    print()

In [ ]:
# ── Roofline plot ──────────────────────────────────────────────────────────────
# RTX 4070 SUPER specs
PEAK_TFLOPS_FP32  = 48.0    # FP32 peak
PEAK_BANDWIDTH_GBS = 432.0  # GB/s

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_title("Roofline Model — RTX 4070 SUPER (FP32)", fontsize=13, fontweight="bold")

# Draw roofline
ai_range = np.logspace(-2, 3, 500)
memory_roof  = PEAK_BANDWIDTH_GBS * ai_range / 1e3   # TFLOPS (bandwidth * AI)
compute_roof = np.full_like(ai_range, PEAK_TFLOPS_FP32)
roofline = np.minimum(memory_roof, compute_roof)

ax.loglog(ai_range, roofline, "k-", lw=2.5, label="Roofline (RTX 4070 SUPER)")
ax.axhline(PEAK_TFLOPS_FP32, color="gray", linestyle="--", alpha=0.5,
           label=f"Peak compute: {PEAK_TFLOPS_FP32} TFLOPS")

# Ridge point
ridge_ai = PEAK_TFLOPS_FP32 * 1e3 / PEAK_BANDWIDTH_GBS
ax.axvline(ridge_ai, color="orange", linestyle=":", alpha=0.7,
           label=f"Ridge point: {ridge_ai:.1f} FLOP/byte")

# Plot our profiled ops
colors  = ["#e74c3c", "#2ecc71", "#3498db"]
markers = ["o", "s", "^"]
for prof, color, marker in zip([prof_gemm, prof_conv, prof_attn], colors, markers):
    ai   = prof.arithmetic_intensity
    perf = prof.tflops
    ax.scatter(ai, perf, color=color, s=150, marker=marker, zorder=5,
               label=f"{prof.name.split('(')[0].strip()} ({ai:.1f} FLOP/B, {perf:.2f} TFLOPS)")
    ax.annotate(prof.name.split("(")[0].strip(),
                xy=(ai, perf), xytext=(ai * 1.3, perf * 1.2),
                fontsize=8, arrowprops=dict(arrowstyle="->", color=color), color=color)

ax.set_xlabel("Arithmetic Intensity (FLOP / byte)", fontsize=11)
ax.set_ylabel("Performance (TFLOPS)", fontsize=11)
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, which="both", alpha=0.3)
ax.set_xlim(0.01, 1000)
ax.set_ylim(0.001, 200)

plt.tight_layout()
plt.show()

print("Interpretation:")
for prof in [prof_gemm, prof_conv, prof_attn]:
    ai = prof.arithmetic_intensity
    bound = "COMPUTE-bound" if ai > ridge_ai else "MEMORY-bound"
    print(f"  {prof.name.split('(')[0].strip():40s} AI={ai:6.1f} → {bound}")

## 5. 📊 Benchmarking Inference — Latency vs Throughput

Inference performance is a two-dimensional tradeoff:

```
  LATENCY = time per request          THROUGHPUT = requests per second

  batch_size=1                        batch_size=64
  ┌────────────────────┐              ┌──────────────────────────────────┐
  │  Request arrives   │              │  Batch 64 requests, wait until   │
  │  Process 1 sample  │              │  full, process all together.     │
  │  Return in 2 ms    │              │  Return in 12 ms — but you've    │
  └────────────────────┘              │  processed 64/12ms = 5333 req/s  │
  Latency-optimised                   └──────────────────────────────────┘
  (interactive use: chatbots)         Throughput-optimised
                                      (offline use: batch inference)
```

### torch.compile vs eager mode
PyTorch 2.0's `torch.compile` uses TorchDynamo to capture the graph and TorchInductor
to generate optimized Triton kernels. The first call triggers compilation (slow), but
subsequent calls are significantly faster due to:
- **Kernel fusion** — adjacent elementwise ops merged into one kernel
- **Memory planning** — optimal buffer allocation
- **Backend specialization** — kernels tuned for your specific GPU

In [ ]:
# ── BenchmarkResult dataclass ─────────────────────────────────────────────────
# Same pattern as notebook 04 — reusable across all benchmarking tasks.

@dataclass
class BenchmarkResult:
    name: str
    batch_size: int
    times_ms: List[float]

    @property
    def mean_ms(self):    return float(np.mean(self.times_ms))
    @property
    def median_ms(self):  return float(np.median(self.times_ms))
    @property
    def std_ms(self):     return float(np.std(self.times_ms))
    @property
    def p95_ms(self):     return float(np.percentile(self.times_ms, 95))
    @property
    def p99_ms(self):     return float(np.percentile(self.times_ms, 99))
    @property
    def throughput(self): return self.batch_size / (self.median_ms * 1e-3)

    def report(self):
        print(f"  {self.name} (batch={self.batch_size})")
        print(f"    Median latency: {self.median_ms:.3f} ms")
        print(f"    P99 latency   : {self.p99_ms:.3f} ms")
        print(f"    Throughput    : {self.throughput:,.0f} samples/sec")


def benchmark_inference(model, batch_size, input_shape, n_warmup=20, n_runs=100,
                         device="cuda", name=None):
    """Benchmark model inference with CUDA events."""
    model = model.to(device).eval()
    x = torch.randn(batch_size, *input_shape, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(x)
    torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            start = torch.cuda.Event(enable_timing=True)
            end   = torch.cuda.Event(enable_timing=True)
            start.record()
            _ = model(x)
            end.record()
            torch.cuda.synchronize()
            times.append(start.elapsed_time(end))

    return BenchmarkResult(
        name=name or "model",
        batch_size=batch_size,
        times_ms=times,
    )


# ── A slightly larger model for meaningful benchmarking ────────────────────────
class BenchMLP(nn.Module):
    def __init__(self, d=512, depth=6):
        super().__init__()
        layers = []
        for i in range(depth):
            layers += [nn.Linear(d, d), nn.GELU()]
        layers.append(nn.Linear(d, d))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


bench_model = BenchMLP(d=512, depth=6).cuda().eval()

# ── Eager vs torch.compile ─────────────────────────────────────────────────────
print("Compiling model with torch.compile (mode='reduce-overhead') ...")
compiled_model = torch.compile(bench_model, mode="reduce-overhead")
# Trigger compilation with a dummy forward pass
with torch.no_grad():
    _ = compiled_model(torch.randn(32, 512, device="cuda"))
torch.cuda.synchronize()
print("Compilation done.")

print()
r_eager    = benchmark_inference(bench_model,    batch_size=32, input_shape=(512,), name="Eager")
r_compiled = benchmark_inference(compiled_model, batch_size=32, input_shape=(512,), name="torch.compile")

print("Results:")
print("-" * 50)
r_eager.report()
r_compiled.report()

speedup = r_eager.median_ms / r_compiled.median_ms
print(f"\n  torch.compile speedup: {speedup:.2f}×")

In [ ]:
# ── Batch size sweep: latency vs throughput tradeoff ──────────────────────────
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]
eager_results    = []
compiled_results = []

print("Batch size sweep (eager vs compiled):")
print(f"{'Batch':>8}  {'Eager lat':>12}  {'Compiled lat':>14}  {'Eager tput':>12}  {'Compiled tput':>14}  {'Speedup':>8}")
print("-" * 80)

for bs in batch_sizes:
    re = benchmark_inference(bench_model,    batch_size=bs, input_shape=(512,),
                              n_warmup=10, n_runs=50, name=f"Eager bs={bs}")
    rc = benchmark_inference(compiled_model, batch_size=bs, input_shape=(512,),
                              n_warmup=10, n_runs=50, name=f"Compiled bs={bs}")
    eager_results.append(re)
    compiled_results.append(rc)
    speedup = re.median_ms / rc.median_ms
    print(f"{bs:>8}  {re.median_ms:>10.3f}ms  {rc.median_ms:>12.3f}ms  "
          f"{re.throughput:>10,.0f}/s  {rc.throughput:>12,.0f}/s  {speedup:>7.2f}×")

In [ ]:
# ── Visualise batch size sweep ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Inference Benchmark — Eager vs torch.compile", fontsize=13, fontweight="bold")

# Left: latency vs batch size
ax = axes[0]
ax.set_title("Latency vs Batch Size")
ax.plot(batch_sizes, [r.median_ms for r in eager_results],
        marker="o", label="Eager",          color="#e74c3c", lw=2)
ax.plot(batch_sizes, [r.median_ms for r in compiled_results],
        marker="s", label="torch.compile",  color="#2ecc71", lw=2)
ax.set_xlabel("Batch size")
ax.set_ylabel("Median latency (ms)")
ax.set_xscale("log", base=2)
ax.legend()
ax.grid(True, alpha=0.3)

# Right: throughput vs batch size
ax2 = axes[1]
ax2.set_title("Throughput vs Batch Size")
ax2.plot(batch_sizes, [r.throughput / 1000 for r in eager_results],
         marker="o", label="Eager",          color="#e74c3c", lw=2)
ax2.plot(batch_sizes, [r.throughput / 1000 for r in compiled_results],
         marker="s", label="torch.compile",  color="#2ecc71", lw=2)
ax2.set_xlabel("Batch size")
ax2.set_ylabel("Throughput (k samples/sec)")
ax2.set_xscale("log", base=2)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_eager    = max(eager_results,    key=lambda r: r.throughput)
best_compiled = max(compiled_results, key=lambda r: r.throughput)
print(f"Peak eager throughput    : {best_eager.throughput:,.0f} samples/s at batch_size={best_eager.batch_size}")
print(f"Peak compiled throughput : {best_compiled.throughput:,.0f} samples/s at batch_size={best_compiled.batch_size}")

## 6. 🔀 Model Sharding for Multi-GPU Deployment

Large language models (GPT-3: 175B params, Llama 3: 70B params) don't fit on a single
GPU. NVIDIA engineers split them across multiple GPUs using two strategies:

### Tensor Parallelism (TP)
Split individual weight matrices across GPUs along rows or columns.
Each GPU computes a partial result; an AllReduce collective combines them.

```
  Full linear layer (d_in=8, d_out=4):
  ┌─────────────────────────────────┐
  │  W  [8 × 4]                    │
  └─────────────────────────────────┘

  Column-parallel (split output dim across 2 GPUs):
  GPU 0: W0 [8 × 2]    GPU 1: W1 [8 × 2]
  y0 = x @ W0          y1 = x @ W1
  ──── AllGather ──────────────────────▶  y = [y0 | y1]

  Row-parallel (split input dim across 2 GPUs — used for second linear in FFN):
  GPU 0: W0 [4 × 4],  x0 = x[:4]    GPU 1: W1 [4 × 4],  x1 = x[4:]
  y0 = x0 @ W0                       y1 = x1 @ W1
  ──── AllReduce ──────────────────────▶  y = y0 + y1
```

### Pipeline Parallelism (PP)
Split the model's layers across GPUs sequentially.
GPU 0 runs layers 0–11, GPU 1 runs layers 12–23, etc.
Micro-batching keeps all GPUs busy by overlapping compute and communication.

```
  Layers  0– 7 → GPU 0  ──┐
  Layers  8–15 → GPU 1  ──┤ forward
  Layers 16–23 → GPU 2  ──┤
  Layers 24–31 → GPU 3  ──┘

  With micro-batches (GPipe schedule):
  Time →
  GPU0: [mb1] [mb2] [mb3] [mb4]  ...
  GPU1:       [mb1] [mb2] [mb3]  ...
  GPU2:             [mb1] [mb2]  ...
  GPU3:                   [mb1]  ...
  All GPUs computing simultaneously after initial pipeline fill.
```

### Combining: 3D Parallelism
Large clusters (512+ GPUs) use **TP + PP + Data Parallelism** simultaneously,
which is how GPT-4 and similar models are trained and served.

In [ ]:
# ── Simulate tensor parallelism on a single GPU ───────────────────────────────
# We split a large linear layer's weight matrix into N shards and show that
# the partitioned computation produces the same result as the original.
# On a real multi-GPU system, each shard would live on a different GPU.

torch.manual_seed(0)

D_IN  = 512
D_OUT = 256
BS    = 8      # batch size
N_GPUS = 4     # number of GPUs to simulate

# Original full-size linear layer
W_full = torch.randn(D_OUT, D_IN)   # (out, in) — PyTorch convention
b_full = torch.randn(D_OUT)
x      = torch.randn(BS, D_IN)

# Reference output
y_ref = F.linear(x, W_full, b_full)   # (BS, D_OUT)

# ── Column-parallel split (split D_OUT across GPUs) ───────────────────────────
shard_size = D_OUT // N_GPUS
W_shards = W_full.split(shard_size, dim=0)   # each (shard_size, D_IN)
b_shards = b_full.split(shard_size, dim=0)   # each (shard_size,)

y_shards = []
for gpu_id, (W_shard, b_shard) in enumerate(zip(W_shards, b_shards)):
    y_partial = F.linear(x, W_shard, b_shard)   # (BS, shard_size)
    y_shards.append(y_partial)
    print(f"  GPU {gpu_id}: W_shard shape={tuple(W_shard.shape)}, output shape={tuple(y_partial.shape)}")

# AllGather: concatenate partial outputs along the output dimension
y_parallel = torch.cat(y_shards, dim=1)   # (BS, D_OUT)

max_diff = (y_ref - y_parallel).abs().max().item()
print(f"\n  AllGather → y_parallel shape: {tuple(y_parallel.shape)}")
print(f"  Max diff vs reference: {max_diff:.2e}  ", end="")
print("✅ Column-parallel is mathematically equivalent" if max_diff < 1e-5 else "❌ Mismatch")

# ── Row-parallel split (split D_IN across GPUs) ───────────────────────────────
print()
W_row_shards = W_full.split(D_IN // N_GPUS, dim=1)   # each (D_OUT, D_IN/N)
x_shards     = x.split(D_IN // N_GPUS, dim=1)         # each (BS, D_IN/N)

y_row_shards = []
for gpu_id, (W_shard, x_shard) in enumerate(zip(W_row_shards, x_shards)):
    y_partial = F.linear(x_shard, W_shard, bias=None)   # (BS, D_OUT) — no bias on partials
    y_row_shards.append(y_partial)
    print(f"  GPU {gpu_id}: W_shard shape={tuple(W_shard.shape)}, x_shard={tuple(x_shard.shape)}, output={tuple(y_partial.shape)}")

# AllReduce: sum partial outputs, then add bias once
y_row_parallel = sum(y_row_shards) + b_full   # (BS, D_OUT)

max_diff2 = (y_ref - y_row_parallel).abs().max().item()
print(f"\n  AllReduce → y_row_parallel shape: {tuple(y_row_parallel.shape)}")
print(f"  Max diff vs reference: {max_diff2:.2e}  ", end="")
print("✅ Row-parallel is mathematically equivalent" if max_diff2 < 1e-5 else "❌ Mismatch")

In [ ]:
# ── Pipeline parallelism simulation ───────────────────────────────────────────
# Split a deep MLP into pipeline stages; verify correctness.

class PipelineStage(nn.Module):
    """One stage of a pipeline-parallel model."""
    def __init__(self, d, n_layers, stage_id):
        super().__init__()
        self.stage_id = stage_id
        layers = []
        for _ in range(n_layers):
            layers += [nn.Linear(d, d), nn.GELU()]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


D, LAYERS_PER_STAGE, N_STAGES = 128, 3, 4

# Build monolithic model and pipeline-split model with the SAME weights
torch.manual_seed(99)
stages = [PipelineStage(D, LAYERS_PER_STAGE, i) for i in range(N_STAGES)]

# Reference: run all stages sequentially on single "device"
x_pp = torch.randn(16, D)
h = x_pp
stage_outputs = []
for stage_id, stage in enumerate(stages):
    h = stage(h)
    stage_outputs.append(h.shape)
    print(f"  Stage {stage_id} (GPU {stage_id}): input → output shape = {tuple(stage_outputs[-1])}")

y_pipeline = h
print(f"\n  Final pipeline output shape: {tuple(y_pipeline.shape)}")
print(f"  Total layers in pipeline   : {N_STAGES * LAYERS_PER_STAGE} × Linear+GELU")

# Show micro-batch scheduling (conceptual)
print()
print("Micro-batch pipeline schedule (4 GPUs, 4 micro-batches):")
print("-" * 60)
N_MICROBATCH = 4
header = f"{'':6}" + "".join(f" t={t:<3}" for t in range(N_STAGES + N_MICROBATCH - 1))
print(header)
for s in range(N_STAGES):
    row = f"GPU {s}: "
    for t in range(N_STAGES + N_MICROBATCH - 1):
        mb = t - s
        if 0 <= mb < N_MICROBATCH:
            row += f" mb{mb}  "
        else:
            row += "  --  "
    print(row)

fill_pct = (N_MICROBATCH * N_STAGES) / ((N_STAGES + N_MICROBATCH - 1) * N_STAGES) * 100
print(f"\n  Pipeline utilization (GPipe, no interleaving): {fill_pct:.1f}%")
print(f"  Idle time ('pipeline bubble'): {100 - fill_pct:.1f}%")

---
## 🎯 Challenge 1: Quantize a Transformer Block and Measure Accuracy vs Speed

You are given a small transformer-style feed-forward block.
Your tasks:

1. **Implement per-channel symmetric INT8 quantization** for the weight matrices.
   Per-channel means each output neuron gets its own scale (more accurate than per-tensor).
   ```
   scale[i] = max(|W[i, :]|) / 127   for each output channel i
   ```
2. **Apply it** to the `TransformerFFN` below by replacing both linear layers.
3. **Measure** output MAE compared to FP32 on 100 random batches.
4. **Benchmark** inference time of FP32 vs per-tensor INT8 vs per-channel INT8
   using `benchmark_inference()`. Report latency and throughput at batch_size=32.
5. **Print a summary table** with columns: Method | MAE | Latency (ms) | Throughput | Memory (KB)

**Expected insight:** per-channel quantization has lower MAE than per-tensor at minimal
additional overhead because the scale better matches each neuron's dynamic range.

### 🧠 Real engineering context
This is exactly what TensorRT does internally when you set `INT8` precision mode.
On Tensor Core hardware the actual speedup comes from running INT8 GEMMs, which give
~4× more throughput than FP32 GEMMs. Our fake-quantization simulation shows the
accuracy side; production deployments combine this with `torch.ao.quantization` or
TensorRT calibration.

In [ ]:
# ── Given: a small transformer FFN block ──────────────────────────────────────
class TransformerFFN(nn.Module):
    """Two-layer feed-forward block, as found in GPT/BERT."""
    def __init__(self, d_model=256, d_ff=1024):
        super().__init__()
        self.fc1  = nn.Linear(d_model, d_ff)
        self.act  = nn.GELU()
        self.fc2  = nn.Linear(d_ff, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(self.fc2(self.act(self.fc1(x))) + x)


torch.manual_seed(0)
ffn_fp32 = TransformerFFN(d_model=256, d_ff=1024).eval()


# ✏️ YOUR TASK
# Step 1: Implement per-channel symmetric INT8 quantization
def quantize_per_channel_int8(tensor: torch.Tensor):
    """
    Per-channel (per-output-neuron) symmetric INT8 quantization.
    tensor shape: (out_features, in_features)
    Returns: (q_int8, scales) where scales has shape (out_features,)
    """
    # ✏️ Implement here
    pass


# Step 2: Create a QuantizedLinearPerChannel module (similar to QuantizedLinear above
#         but using per-channel scales)
class QuantizedLinearPerChannel(nn.Module):
    def __init__(self, linear: nn.Linear):
        super().__init__()
        # ✏️ Implement here
        pass

    def forward(self, x):
        # ✏️ Implement here
        pass


# Step 3: Build the three variants
# ffn_per_tensor  = ...   # use quantize_model() from section 2
# ffn_per_channel = ...   # use your new QuantizedLinearPerChannel


# Step 4: Measure MAE on 100 batches
# ✏️ YOUR CODE HERE


# Step 5: Benchmark latency
# ✏️ YOUR CODE HERE


# Step 6: Print summary table
# ✏️ YOUR CODE HERE
print("Challenge 1: implement the cells above and re-run.")

---
## 🎯 Challenge 2: Profile a Model and Identify the Top-3 Bottleneck Ops

You are given a small vision-style model with several different layer types.
Your tasks:

1. **Profile each layer individually** using `profile_op()` from Section 4.
   For each layer, you need to compute the theoretical FLOPs and bytes accessed.
2. **Rank layers by total latency** (median_ms).
3. **Identify the top-3 bottleneck layers** and state whether each is
   compute-bound or memory-bound (using the roofline ridge point).
4. **Suggest one optimization** for each bottleneck:
   - Compute-bound: reduce FLOPs (pruning, lower rank), or use Tensor Cores (FP16)
   - Memory-bound: fuse with adjacent ops, increase batch size, or quantize

### 🧠 Real engineering context
This is the core of NVIDIA's inference optimization workflow. Tools like
**Nsight Systems** (`nsys profile`) and **Nsight Compute** (`ncu`) give you the
same information at kernel level. Understanding which ops dominate, and why,
is what separates a senior DL engineer from a junior one.

In [ ]:
# ── Given: a small vision model with mixed layer types ────────────────────────
# Your job is to profile each block and find the bottlenecks.

BS_C2 = 16   # batch size for profiling

# Layer definitions — profile each one separately
layers_to_profile = {
    "conv_stem":    (nn.Conv2d(3, 64, 7, stride=2, padding=3).cuda(),
                     (BS_C2, 3, 224, 224)),
    "conv_block1":  (nn.Conv2d(64, 128, 3, padding=1).cuda(),
                     (BS_C2, 64, 56, 56)),
    "conv_block2":  (nn.Conv2d(128, 256, 3, padding=1).cuda(),
                     (BS_C2, 128, 28, 28)),
    "linear_head":  (nn.Linear(256, 1000).cuda(),
                     (BS_C2, 256)),
    "layernorm":    (nn.LayerNorm(512).cuda(),
                     (BS_C2, 512, 512)),
    "gelu":         (nn.GELU().cuda(),
                     (BS_C2, 1024, 512)),
}

# ✏️ YOUR TASK
# For each (layer, input_shape) pair:
#   1. Compute theoretical FLOPs and bytes accessed
#      (Use the formulas from Section 4 as reference)
#   2. Call profile_op() to get timing
#   3. Collect KernelProfile objects in a list

# profiles = []
# for name, (layer, input_shape) in layers_to_profile.items():
#     x = torch.randn(*input_shape, device="cuda")
#     flops = ...       # ✏️ compute for each layer type
#     bytes_ = ...      # ✏️ compute for each layer type
#     p = profile_op(name, lambda: layer(x), flops=flops, bytes_accessed=bytes_)
#     profiles.append(p)

# ✏️ Step 2: Sort by median_ms and print a ranked table

# ✏️ Step 3: Identify top-3 bottlenecks and classify as compute/memory bound

# ✏️ Step 4: Suggest one optimization for each

print("Challenge 2: implement the profiling loop above and re-run.")

---
## 🎯 Challenge 3: Batch Size Sweep and Throughput Curve Analysis

You are given a small attention-based model. Your tasks:

1. **Implement a batch size sweep** from 1 to 512 (powers of 2) using `benchmark_inference()`.
2. **Plot** two curves side-by-side:
   - Left: Latency (ms) vs batch size — log scale on x-axis
   - Right: Throughput (samples/sec) vs batch size — log scale on x-axis
3. **Find and mark** the "knee point" — the batch size where throughput growth flattens
   (i.e., the marginal throughput gain per doubling drops below 20%).
4. **Compute and print** the GPU utilization estimate:
   ```
   utilization = achieved_TFLOPS / peak_TFLOPS_FP32 * 100
   ```
   Use the roofline formulas from Section 4. Compute FLOPS for one forward pass
   of `AttentionModel` (count matmuls only), multiply by throughput.
5. **State** at which batch size the GPU is most efficiently utilized and why.

### 🧠 Real engineering context
This analysis tells you the optimal batch size for serving. For an online API
you want the knee point or below (predictable latency). For offline inference
you want maximum batch size where the GPU is saturated. NVIDIA's TensorRT-LLM
uses dynamic batching to pack requests to stay near this optimal operating point.

In [ ]:
# ── Given: a small multi-head self-attention model ────────────────────────────

class AttentionModel(nn.Module):
    """Single transformer block: MHSA + FFN + residuals."""
    def __init__(self, d_model=256, n_heads=8, seq_len=64, d_ff=1024):
        super().__init__()
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.seq_len  = seq_len
        self.d_head   = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        B, S, D = x.shape
        # MHSA
        qkv = self.qkv_proj(x).reshape(B, S, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)  # (B, H, S, D_h)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(q, k, v)
        attn_out = attn_out.transpose(1, 2).reshape(B, S, D)
        x = self.norm1(x + self.out_proj(attn_out))
        # FFN
        x = self.norm2(x + self.ffn(x))
        return x


attn_model = AttentionModel(d_model=256, n_heads=8, seq_len=64, d_ff=1024).cuda().eval()
print(f"AttentionModel params: {sum(p.numel() for p in attn_model.parameters()):,}")

# Input shape: (batch_size, seq_len=64, d_model=256)
INPUT_SHAPE = (64, 256)   # (seq_len, d_model)

# ✏️ YOUR TASK
# Step 1: Run batch size sweep
batch_sizes_c3 = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
# results_c3 = []
# for bs in batch_sizes_c3:
#     r = benchmark_inference(attn_model, batch_size=bs, input_shape=INPUT_SHAPE,
#                              n_warmup=10, n_runs=50, name=f"Attention bs={bs}")
#     results_c3.append(r)
#     print(f"  bs={bs:>4}  latency={r.median_ms:.3f}ms  throughput={r.throughput:>10,.0f}/s")

# Step 2: Plot latency and throughput curves
# ✏️ YOUR CODE HERE

# Step 3: Find the knee point
# ✏️ YOUR CODE HERE

# Step 4: Compute GPU utilization
# ✏️ Compute FLOPs for one forward pass of AttentionModel at batch_size=32:
#   - QKV projection: 2 * BS * S * D * 3D
#   - Attention: 2 * BS * H * S * S * D_h (QK^T) + 2 * BS * H * S * S * D_h (AV)
#   - Out projection: 2 * BS * S * D * D
#   - FFN fc1: 2 * BS * S * D * D_ff
#   - FFN fc2: 2 * BS * S * D_ff * D
# utilization = achieved_tflops / 48.0 * 100   (RTX 4070 SUPER FP32 peak)

# Step 5: Print your conclusion
# ✏️ YOUR CODE HERE

print("Challenge 3: uncomment and implement the cells above, then re-run.")

---
## ✅ Module Summary

| Skill | What you learned | Job requirement |
|---|---|---|
| **PyTorch 2.0 IR** | torch.fx graph capture, torch.export, node inspection | "PyTorch 2.0 ecosystem — torch.compile, torch.export" |
| **INT8 quantization** | Scale/zero_point math, per-tensor vs per-channel, fake-quant | "Model optimization: quantization" |
| **FP16 mixed precision** | Memory savings, dtype cast, error analysis | "Train and deploy generative AI models" |
| **Magnitude pruning** | `torch.nn.utils.prune`, global unstructured, sparsity sweep | "Pruning, sparsity, neural architecture search" |
| **CUDA event profiling** | `torch.cuda.Event`, kernel timing, FLOPS/bandwidth | "GPU kernel-level performance analysis (CUDA)" |
| **Roofline model** | Arithmetic intensity, compute vs memory bound | "GPU kernel-level performance analysis" |
| **torch.compile** | Eager vs compiled benchmark, reduce-overhead mode | "PyTorch 2.0 ecosystem — torch.compile" |
| **Latency vs throughput** | Batch size sweep, BenchmarkResult dataclass | "Scalable deployment" |
| **Tensor parallelism** | Column-parallel / row-parallel weight splits, AllGather, AllReduce | "Automated model sharding, scalable deployment" |
| **Pipeline parallelism** | Stage scheduling, micro-batching, bubble analysis | "Scalable deployment" |

---

### Next steps for this role
- **TensorRT** — convert a PyTorch model to a TensorRT engine and benchmark INT8 speedup
- **Triton** — write a custom fused kernel (e.g., fused layer norm + GELU) in Triton
- **CUTLASS** — understand the GEMM tiling hierarchy (thread block → warp → thread)
- **Flash Attention** — study the tiled memory-efficient attention algorithm
- **GPTQ / AWQ** — explore LLM-specific weight-only INT4 quantization
- **Nsight Systems** — profile a real model end-to-end and read the timeline

### Resources
- [TensorRT Developer Guide](https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/)
- [Triton tutorials](https://triton-lang.org/main/getting-started/tutorials/)
- [CUTLASS documentation](https://github.com/NVIDIA/cutlass)
- [torch.compile deep dive](https://pytorch.org/docs/stable/torch.compiler.html)
- [torch.ao.quantization](https://pytorch.org/docs/stable/quantization.html)
- [PyTorch 2.0 export docs](https://pytorch.org/docs/stable/export.html)
- [Flash Attention paper](https://arxiv.org/abs/2205.14135)
- [Megatron-LM (tensor/pipeline parallelism)](https://github.com/NVIDIA/Megatron-LM)